In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
# df = pd.DataFrame()

# for file in files_:
#     tmp = pd.read_excel(f'{files_path}/{file}', sheet_name=None)
#     print(f'{file}: {tmp.keys()}')

#     if 'Sheet1' in tmp.keys():
#         file_df = tmp['Sheet1']
#         file_df.columns = file_df.columns.str.lower()
#     else:
#         file_df = tmp['Base']
#         file_df.columns = file_df.columns.str.lower()

#     file_df['file'] = file
#     df = pd.concat([df, file_df], ignore_index=True)

df = pd.read_excel(f"/data/aman_singh/acuuracy_check/SOH Data - 01 Aug.xlsx", sheet_name='Base')

In [4]:
df.columns

Index([         'Date',         'Chain',           'FSN',           'SOH',
       'material_code',          'Desc',         'Brand',           'UOM',
        'vol per unit',           'Vol',      'FY INDEX',     'Vol in KL',
         'BPM in lacs',      'Category',   'eCOM Brands',          'PSKU',
                'PDES',      'Club SKU',   'Unnamed: 18',   'Unnamed: 19',
         'Unnamed: 20',   'Unnamed: 21',            1000],
      dtype='object')

In [7]:
df.columns = df.columns.str.lower()

In [8]:
df

,date,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,...,category,ecom brands,psku,pdes,club sku,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,NaN
0,2026-08-01,Big Basket,91432,14.0,707611,SW HrGel WL 10ml Tube SPT,SW HRGEL,L,10.0,0.1400,...,Male Grooming,eCOM Brands,718521.0,SETWET HAIRGEL WET LOOK 10ml PCH,10ml,NaN,NaN,NaN,NaN,NaN
1,2026-08-01,Big Basket,100554,8325.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,900.0,7.4925,...,Edible,eCOM Brands,718328.0,SAFF TASTY 1L PCH,1LTR PC,NaN,NaN,NaN,NaN,NaN
2,2026-08-01,Big Basket,126153,1611.0,713332,SAF TASTY PLUS 5L JAR,SAFF KOCO,KL,5000.0,8.0550,...,Edible,eCOM Brands,718330.0,SAFF TASTY 5L JAR,5LTR,NaN,NaN,NaN,NaN,NaN
3,2026-08-01,Big Basket,147491,13212.0,716968,SAF GOLD 1L PCH (SP)-PL,SAFF GOLD,KL,1000.0,13.2120,...,Edible,eCOM Brands,718341.0,SAFF GOLD 1L PCH,1LTR PC,NaN,NaN,NaN,NaN,NaN
4,2026-08-01,Big Basket,147492,2038.0,716965,SAF GOLD 2L JAR (SP)-PL,SAFF GOLD,KL,2000.0,4.0760,...,Edible,eCOM Brands,718342.0,SAFF GOLD 2L JAR,2LTR,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2952,2026-08-17,Flipkart Grocery,TLCHFK7WEJ6HKNYZ,4.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,0.8000,...,PA BABY,eCOM Brands,809046.0,PAR ADV BABY POWDER 200 GMS,200GM,NaN,NaN,NaN,NaN,NaN
2953,2026-08-17,Flipkart Minutes,TLCHFK7WEJ6HKNYZ,1062.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,212.4000,...,PA BABY,eCOM Brands,809046.0,PAR ADV BABY POWDER 200 GMS,200GM,NaN,NaN,NaN,NaN,NaN
2954,2026-08-17,Flipkart Grocery,WIPHBWADB6FNPEJH,29.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,10.4980,...,PA BABY,eCOM Brands,810738.0,PA BABY FACE BODY WIPE 362GM,362GM,NaN,NaN,NaN,NaN,NaN
2955,2026-08-17,Flipkart Minutes,WIPHBWADB6FNPEJH,285.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,103.1700,...,PA BABY,eCOM Brands,810738.0,PA BABY FACE BODY WIPE 362GM,362GM,NaN,NaN,NaN,NaN,NaN


In [9]:
soh_base = pd.read_csv('/data/aman_singh/acuuracy_check/soh_base_jul_run.csv')
soh_base

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,roum,roum divide,month,day,month.1,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,Unnamed: 41
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89874,Amazon RK,B0GRVG1BH2,969.0,811241,PA ROSMARY HAIR SRAY 100ML,PA_RSW_SR,L,100.0,96.900000,740.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89875,Amazon RK,B0GT8SRWNG,69.0,811199,PA PEPPERMINT ESS OIL 14ML,PA_ESS_HO,L,14.0,0.966000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89876,Amazon RK,B0GT8ZGZ4T,172.0,811197,PA ENCALYPTUS ESS OIL 14ML,PA_ESS_HO,L,14.0,2.408000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89877,Amazon RK,B0GT91H554,118.0,811198,PA SANDALWOOD ESS OIL 14ML,PA_ESS_HO,L,14.0,1.652000,12860.631072,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
soh_base.columns

Index(['chain', 'fsn', 'soh', 'material_code', 'desc', 'brand', 'uom',
       'vol per unit', 'vol', 'fy index', 'bpm in lacs', 'category',
       'ecom brands', 'file', 'vol in kl', 'psku', 'pdes', 'date', 'asin',
       'asin.1', 'platform_name.1', 'product_title', 'ean', 'units',
       'material_group_code', 'uom_reporting', 'vol_per_unit', 'vol_in_lit',
       'vol in rum', 'indexrate', 'indexbpm', 'club sku', 'roum',
       'roum divide', 'month', 'day', 'month.1', 'unnamed: 18', 'unnamed: 19',
       'unnamed: 20', 'unnamed: 21', 'Unnamed: 41'],
      dtype='object')

In [11]:
df['file'] = 'SOH - 01 Aug.xlsx'

In [12]:
final_df = pd.concat([soh_base, df], ignore_index=True)
final_df

,chain,fsn,soh,material_code,desc,brand,uom,vol per unit,vol,fy index,...,roum divide,month,day,month.1,unnamed: 18,unnamed: 19,unnamed: 20,unnamed: 21,Unnamed: 41,NaN
0,Blinkit,10000059,3131.0,707136,SAF TOTAL 5LT JAR,SAFF KO,KL,5000.0,15.655000,182263.023590,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Blinkit,10000062,500.0,722897,SMO 38G CUR&PEP HDC,SFOATS-FL,TO,38.0,0.019000,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Blinkit,10000063,2711.0,722898,SMO 38G MAS&COR HDC,SFOATS-FL,TO,38.0,0.103018,252810.051419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Blinkit,10000361,1542.0,809267,SAFF SALT PLUS 1 KG PCH,SAFF SALT,TO,1000.0,1.542000,25994.993781,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Blinkit,10000362,3387.0,713333,SAF TASTY PLUS 1L PCH,SAFF KOCO,KL,1000.0,3.387000,145305.652227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92831,Flipkart Grocery,TLCHFK7WEJ6HKNYZ,4.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,0.800000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92832,Flipkart Minutes,TLCHFK7WEJ6HKNYZ,1062.0,809047,PA BABY POWDER 200 GMS,PABABY_GM,KG,200.0,212.400000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92833,Flipkart Grocery,WIPHBWADB6FNPEJH,29.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,10.498000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92834,Flipkart Minutes,WIPHBWADB6FNPEJH,285.0,810688,PA BABY FACE BDY WIPE 362G,PABABY_GM,KG,362.0,103.170000,366.484998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
final_df['date'] = pd.to_datetime(final_df['date'])
final_df['date'].unique()

<DatetimeArray>
['2025-04-01 00:00:00', '2025-03-28 00:00:00', '2025-03-11 00:00:00',
 '2025-03-29 00:00:00', '2025-03-23 00:00:00', '2025-03-22 00:00:00',
 '2025-03-31 00:00:00', '2025-09-05 00:00:00', '2025-09-01 00:00:00',
 '2025-08-26 00:00:00',
 ...
 '2026-05-01 00:00:00', '2026-05-03 00:00:00', '2026-04-30 00:00:00',
 '2026-05-04 00:00:00', '2026-04-29 00:00:00', '2026-06-01 00:00:00',
 '2026-06-02 00:00:00', '2026-07-01 00:00:00', '2026-08-01 00:00:00',
 '2026-08-17 00:00:00']
Length: 101, dtype: datetime64[ns]

In [14]:
final_df.to_csv('/data/aman_singh/acuuracy_check/soh_base_aug_run.csv', index=False)

In [17]:
final_df['month_end'] = final_df['date'] + pd.offsets.MonthEnd(0)

In [18]:
final_df.groupby(['month_end'])['soh'].sum()

month_end
2024-12-31    2.695180e+06
2025-01-31    3.063929e+06
2025-02-28    2.716948e+06
2025-03-31    1.670496e+06
2025-04-30    4.745680e+06
2025-05-31    2.740214e+06
2025-06-30    4.214220e+06
2025-07-31    1.971143e+06
2025-08-31    7.356897e+06
2025-09-30    9.830427e+06
2025-10-31    7.956328e+06
2025-11-30    1.127015e+06
2025-12-31    7.708016e+06
2026-01-31    4.790708e+06
2026-02-28    2.795760e+06
2026-03-31    3.647053e+06
2026-04-30    4.082873e+06
2026-05-31    4.148106e+06
2026-06-30    4.404793e+06
Name: soh, dtype: float64

In [14]:
# def clean_date(x):
#     if isinstance(x, str):
#         x = x.replace('st', '').replace('nd', '').replace('rd', '').replace('th', '').strip()
#         # if 'dec' in x.lower():
#         #     x += ' 2024'
#         # else:
#         #     x += ' 2025'
#         x += ' 2025'
#         return pd.to_datetime(x, format='%d %b %Y')
#     else:
#         return pd.to_datetime(x, unit='D', origin='1899-12-30')

In [15]:
# df['date_clean'] = df['date'].map(clean_date)

In [16]:
# df[['date', 'date_clean']].drop_duplicates().to_clipboard(index=False)

In [11]:
# del df['date']

# df.rename(columns={'date_clean': 'date'}, inplace=True)
df['date'] = pd.to_datetime(df['date'])

In [12]:
df.dtypes

date             datetime64[ns]
chain                    object
fsn                      object
soh                       int64
material_code             int64
desc                     object
brand                    object
uom                      object
vol per unit            float64
vol                     float64
fy index                float64
vol in kl               float64
bpm in lacs             float64
category                 object
ecom brands              object
psku                    float64
pdes                     object
club sku                 object
dtype: object

In [13]:
df[['platform_name', 'date']].drop_duplicates().sort_values(by=['platform_name'])

KeyError: "['platform_name'] not in index"

In [20]:
# ###### !!!!!!!!!!!!!!! TEMP #################!!!!!!!!!!!!!!!!!
# print(df[df['date'] > '2025-12-31']['date'].unique())
# df['date'] = pd.to_datetime(df['date'])

# cap_date = pd.to_datetime('2025-12-31')
# df['date'] = df['date'].clip(upper=cap_date)

# print(df[df['date'] > '2025-12-31']['date'].unique())

In [21]:
df['date'].max()

Timestamp('2026-02-02 00:00:00')

In [22]:
df[['platform_name', 'date']].drop_duplicates()

,platform_name,date
0,Flipkart Grocery,2026-01-22
209,Nykaa,2026-01-21
362,Blinkit,2026-01-31
557,Zepto,2026-01-31
761,Meesho,2026-01-31
1134,Big Basket,2026-01-31
1306,Swiggy,2026-01-31
1586,Flipkart National,2026-01-30
2042,Flipkart Minutes,2026-01-30
2287,Myntra,2026-01-28


In [23]:
df.rename(columns={'platform_name': 'chain', 'shipped_unit': 'soh'}, inplace=True)

In [24]:
df.to_csv('Clean Master/SOH - 31-Jan26 1 - processed.csv', index=False)

In [11]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [12]:
df.columns = df.columns.str.upper()
df['DATE'] = df['DATE'].astype(str)

In [13]:
df.dtypes

CHAIN             object
FSN               object
SOH              float64
MATERIAL_CODE     object
DESC              object
BRAND             object
UOM               object
VOL PER UNIT     float64
VOL              float64
FY INDEX         float64
BPM IN LACS      float64
CATEGORY          object
ECOM BRANDS       object
FILE              object
VOL IN KL        float64
PSKU              object
PDES              object
DATE              object
dtype: object

In [14]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(dev_conn, df, 
            table_name = "TRN_ECOM_SOH_FROM_BASE_FILES",
            auto_create_table=True,
            overwrite = False,
)

ArrowInvalid: ("Could not convert '00128d5d-d85d-40ed-b70b-c5fa9f10cac0' with type str: tried to convert to int64", 'Conversion failed for column FSN with type object')